# LRA Ablation: All F1 × F2 Combinations vs Standard Softmax

Tasks: **ListOps · Pathfinder · LRA Text · LRA Image**

Runs every combination of F2 scoring function (7) × F1 aggregator (5) = **35 configs** per task.  
The standard softmax baseline is `f2=full_set` + `f1=restricted_softmax`, marked with `*` in all tables.

`f2=full_set` always uses the deterministic full-support path (no Gibbs sampling).

In [ ]:
# !pip install torch torchvision

In [ ]:
import sys, math, time, random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

GAM_DIR = Path(".").resolve()
if str(GAM_DIR) not in sys.path:
    sys.path.insert(0, str(GAM_DIR))

from general_attention import GeneralAttention, GibbsConfig

def choose_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = choose_device()
print(f"Device: {DEVICE}")

## Shared Architecture

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, ff_mult=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * ff_mult), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim * ff_mult, dim), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, dim, attn, ff_mult=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = attn
        self.norm2 = nn.LayerNorm(dim)
        self.ff    = FeedForward(dim, ff_mult, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class SequenceClassifier(nn.Module):
    """Embedding -> N transformer blocks -> mean-pool -> linear head."""
    def __init__(self, vocab_size, num_classes, max_seq_len, dim, depth,
                 attn_factory, ff_mult=4, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[
            TransformerBlock(dim, attn_factory(dim), ff_mult, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        B, L = x.shape
        pos  = torch.arange(L, device=x.device).unsqueeze(0)
        h    = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        h    = self.norm(self.blocks(h))
        return self.head(h.mean(dim=1))

print("Architecture defined.")

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, max_batches=None):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches: break
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct  += (logits.argmax(-1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device, max_batches=None):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches: break
        x, y   = x.to(device), y.to(device)
        logits  = model(x)
        loss    = criterion(logits, y)
        loss_sum += loss.item() * y.size(0)
        correct  += (logits.argmax(-1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)

print("Training utilities defined.")

## Ablation Grid

In [ ]:
# All valid F1 and F2 types from general_attention.py
F2_TYPES = [
    "full_set",
    "modular_dot",
    "modular_dot_hard_singleton",
    "modular_dot_first_free",
    "logsumexp",
    "dot_repulsion",
    "neural_mlp",
]
F1_TYPES = [
    "mean",
    "mlp_mean",
    "mlp_concat",
    "transformer",
    "restricted_softmax",
]

SOFTMAX_KEY = ("full_set", "restricted_softmax")   # standard softmax baseline

# Shared Gibbs config used for all non-full_set F2 types
GIBBS_CFG = GibbsConfig(beta=1.0, gibbs_steps=32, runs=4)


def build_attn_module(dim, f1_type, f2_type):
    """Build a GeneralAttention module for the given (f1, f2) pair."""
    return GeneralAttention(
        d_model=dim,
        f2_type=f2_type,
        f1_type=f1_type,
        cfg=GIBBS_CFG,          # ignored when f2_type='full_set'
        f1_concat_max_set_size=8,
        f1_concat_hidden=64,
        f2_neural_hidden=64,
    )


def run_ablation(
    task_name,
    train_loader,
    val_loader,
    make_model_fn,           # (f1_type, f2_type) -> nn.Module
    epochs=3,
    lr=3e-4,
    device=DEVICE,
    max_train_batches=None,
    max_val_batches=None,
    skip_f2=None,
    skip_f1=None,
):
    """
    Runs every (f2, f1) combination, returns results dict keyed by (f2, f1).
    Standard softmax = (full_set, restricted_softmax).
    """
    skip_f2 = set(skip_f2 or [])
    skip_f1 = set(skip_f1 or [])
    criterion = nn.CrossEntropyLoss()
    results   = {}

    combos = [(f2, f1) for f2 in F2_TYPES if f2 not in skip_f2
                        for f1 in F1_TYPES if f1 not in skip_f1]
    n = len(combos)

    for idx, (f2, f1) in enumerate(combos, 1):
        tag = " [SOFTMAX BASELINE]" if (f2, f1) == SOFTMAX_KEY else ""
        print(f"[{task_name}] ({idx}/{n}) f2={f2}  f1={f1}{tag}")

        model = make_model_fn(f1, f2).to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        t0 = time.time()
        for epoch in range(1, epochs + 1):
            tr_loss, tr_acc = train_epoch(model, train_loader, opt, criterion, device, max_train_batches)
            vl_loss, vl_acc = evaluate(model, val_loader, criterion, device, max_val_batches)
            sched.step()
        print(f"  -> val_acc={vl_acc:.4f}  val_loss={vl_loss:.4f}  ({time.time()-t0:.1f}s)")

        results[(f2, f1)] = {"val_acc": vl_acc, "val_loss": vl_loss}
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return results


def display_results(task_name, results, skip_f2=None, skip_f1=None):
    """Print a F1 (rows) x F2 (cols) accuracy table."""
    skip_f2 = set(skip_f2 or [])
    skip_f1 = set(skip_f1 or [])
    active_f2 = [f for f in F2_TYPES if f not in skip_f2]
    active_f1 = [f for f in F1_TYPES if f not in skip_f1]

    sm_acc = results.get(SOFTMAX_KEY, {}).get("val_acc", float("nan"))

    print(f"\n{'='*90}")
    print(f"  {task_name}  |  val accuracy  |  softmax baseline (*)  = {sm_acc:.4f}")
    print(f"{'='*90}")

    col_w, f1_w = 27, 22
    # Abbreviate long F2 names to fit
    abbrev = {
        "full_set":                  "full_set",
        "modular_dot":               "mod_dot",
        "modular_dot_hard_singleton": "mod_dot_hard",
        "modular_dot_first_free":    "mod_dot_free",
        "logsumexp":                 "logsumexp",
        "dot_repulsion":             "dot_repul",
        "neural_mlp":                "neural_mlp",
    }
    header = f"{'F1 \\ F2':<{f1_w}}" + "".join(f"{abbrev[f2]:>{col_w}}" for f2 in active_f2)
    print(header)
    print("─" * len(header))

    for f1 in active_f1:
        row = f"{f1:<{f1_w}}"
        for f2 in active_f2:
            key = (f2, f1)
            if key not in results:
                cell = "skip"
            else:
                acc    = results[key]["val_acc"]
                marker = "*" if key == SOFTMAX_KEY else " "
                cell   = f"{acc:.4f}{marker}"
            row += cell.rjust(col_w)
        print(row)

    # Best GAM result
    gam = {k: v["val_acc"] for k, v in results.items() if k != SOFTMAX_KEY}
    if gam:
        best_k = max(gam, key=gam.get)
        best_v = gam[best_k]
        print(f"\nBest GAM: f2={best_k[0]}  f1={best_k[1]}  "
              f"val_acc={best_v:.4f}  delta={best_v - sm_acc:+.4f}")

print("Ablation grid defined.  F2 types:", F2_TYPES)
print("                        F1 types:", F1_TYPES)
print(f"Total combinations per task: {len(F2_TYPES) * len(F1_TYPES)}")

---
## Task 1 — ListOps

Nested list operations `[MAX 4 [MIN 2 3] 1]`. 10-class classification. Max seq len: 2048.

Real data: [LRA GitHub](https://github.com/google-research/long-range-arena) → `lra_release/lra_data/listops-1000/`

In [ ]:
LISTOPS_TOKENS = {
    "[MIN": 0, "[MAX": 1, "[MED": 2, "[SM": 3, "]": 4, "PAD": 5,
    **{str(i): 6 + i for i in range(10)},
}
LISTOPS_VOCAB = len(LISTOPS_TOKENS)  # 20


class ListOpsSynthetic(Dataset):
    def __init__(self, size=2000, seq_len=512, seed=42):
        rng = random.Random(seed)
        tok2id = LISTOPS_TOKENS
        ops = ["[MIN", "[MAX", "[MED", "[SM"]
        self.samples = []

        def gen(depth):
            if depth == 0 or rng.random() < 0.4:
                v = rng.randint(0, 9); return [str(v)], v
            op = rng.choice(ops)
            toks, vals = [op], []
            for _ in range(rng.randint(2, 4)):
                st, sv = gen(depth - 1); toks += st; vals.append(sv)
            toks.append("]")
            r = {"[MIN": min, "[MAX": max, "[MED": lambda v: sorted(v)[len(v)//2]}. \
                get(op, lambda v: sum(v) % 10)(vals)
            return toks, r

        for _ in range(size):
            toks, label = gen(4)
            ids = [tok2id.get(t, tok2id["PAD"]) for t in toks][:seq_len]
            ids += [tok2id["PAD"]] * (seq_len - len(ids))
            self.samples.append((ids, label))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        ids, lbl = self.samples[i]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)


def load_listops_real(data_dir, split="train", seq_len=2048):
    fname = {"train": "basic_train.tsv", "val": "basic_val.tsv", "test": "basic_test.tsv"}[split]
    tok2id = LISTOPS_TOKENS
    samples = []
    with open(Path(data_dir) / fname) as f:
        next(f)
        for line in f:
            lbl_str, seq_str = line.strip().split("\t", 1)
            ids = [tok2id.get(t, tok2id["PAD"]) for t in seq_str.split()][:seq_len]
            ids += [tok2id["PAD"]] * (seq_len - len(ids))
            samples.append((ids, int(lbl_str)))
    class _DS(Dataset):
        def __len__(self): return len(samples)
        def __getitem__(self, i):
            ids, lbl = samples[i]
            return torch.tensor(ids, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)
    return _DS()

print("ListOps dataset ready.")

In [ ]:
# ── ListOps config ────────────────────────────────────────────────────────────
LISTOPS_REAL_DATA_DIR  = None   # set to real data dir to use LRA data
LISTOPS_SEQ_LEN        = 512    # use 2048 for full LRA
LISTOPS_DIM            = 64
LISTOPS_DEPTH          = 2
LISTOPS_EPOCHS         = 3
LISTOPS_BATCH          = 32
LISTOPS_MAX_TRAIN      = 60     # batches per epoch; None = full dataset
LISTOPS_MAX_VAL        = 20
LISTOPS_SKIP_F2        = []     # e.g. ["neural_mlp"] to skip slow variants
LISTOPS_SKIP_F1        = []     # e.g. ["transformer"]

if LISTOPS_REAL_DATA_DIR:
    lo_tr = load_listops_real(LISTOPS_REAL_DATA_DIR, "train", LISTOPS_SEQ_LEN)
    lo_vl = load_listops_real(LISTOPS_REAL_DATA_DIR, "val",   LISTOPS_SEQ_LEN)
else:
    print("Synthetic ListOps (set LISTOPS_REAL_DATA_DIR for real data).")
    lo_tr = ListOpsSynthetic(size=2000, seq_len=LISTOPS_SEQ_LEN, seed=0)
    lo_vl = ListOpsSynthetic(size=500,  seq_len=LISTOPS_SEQ_LEN, seed=99)

lo_train_ldr = DataLoader(lo_tr, batch_size=LISTOPS_BATCH, shuffle=True,  num_workers=0)
lo_val_ldr   = DataLoader(lo_vl, batch_size=LISTOPS_BATCH, shuffle=False, num_workers=0)

def make_listops_model(f1_type, f2_type):
    return SequenceClassifier(
        vocab_size=LISTOPS_VOCAB, num_classes=10,
        max_seq_len=LISTOPS_SEQ_LEN, dim=LISTOPS_DIM, depth=LISTOPS_DEPTH,
        attn_factory=lambda d: build_attn_module(d, f1_type, f2_type),
    )

listops_results = run_ablation(
    "ListOps", lo_train_ldr, lo_val_ldr, make_listops_model,
    epochs=LISTOPS_EPOCHS, max_train_batches=LISTOPS_MAX_TRAIN,
    max_val_batches=LISTOPS_MAX_VAL, skip_f2=LISTOPS_SKIP_F2, skip_f1=LISTOPS_SKIP_F1,
)
display_results("ListOps", listops_results, LISTOPS_SKIP_F2, LISTOPS_SKIP_F1)

---
## Task 2 — Pathfinder

Binary classification: does a path connect two circles in a 32×32 image?  
Images flattened to sequences of length **1024**.

Real data: [LRA GitHub](https://github.com/google-research/long-range-arena) → `lra_release/lra_data/pathfinder32/`

In [ ]:
class PathfinderSynthetic(Dataset):
    def __init__(self, size=2000, img_size=32, seed=0):
        torch.manual_seed(seed)
        seq_len = img_size * img_size
        imgs    = torch.randint(0, 2, (size, seq_len))
        centre  = imgs[:, seq_len//4 : 3*seq_len//4].float().mean(dim=1)
        self.data   = imgs
        self.labels = (centre > 0.5).long()

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], self.labels[i]


def load_pathfinder_real(data_dir, split="train", img_size=32):
    import numpy as np
    from PIL import Image
    split_dir = Path(data_dir) / split
    metadata  = np.load(split_dir / "metadata.npy", allow_pickle=True)
    samples   = []
    for entry in metadata:
        img    = np.array(Image.open(split_dir / entry["image"]).convert("L").resize((img_size, img_size)))
        pixels = (img.flatten() > 128).astype(int)
        samples.append((pixels, int(entry["label"])))
    class _DS(Dataset):
        def __len__(self): return len(samples)
        def __getitem__(self, i):
            px, lbl = samples[i]
            return torch.tensor(px, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)
    return _DS()

print("Pathfinder dataset ready.")

In [ ]:
# ── Pathfinder config ─────────────────────────────────────────────────────────
PATHFINDER_REAL_DATA_DIR = None   # e.g. "/data/lra/pathfinder32"
PATHFINDER_IMG_SIZE      = 32
PATHFINDER_SEQ_LEN       = PATHFINDER_IMG_SIZE ** 2   # 1024
PATHFINDER_DIM           = 64
PATHFINDER_DEPTH         = 2
PATHFINDER_EPOCHS        = 3
PATHFINDER_BATCH         = 32
PATHFINDER_MAX_TRAIN     = 60
PATHFINDER_MAX_VAL       = 20
PATHFINDER_SKIP_F2       = []
PATHFINDER_SKIP_F1       = []

if PATHFINDER_REAL_DATA_DIR:
    pf_tr = load_pathfinder_real(PATHFINDER_REAL_DATA_DIR, "train", PATHFINDER_IMG_SIZE)
    pf_vl = load_pathfinder_real(PATHFINDER_REAL_DATA_DIR, "val",   PATHFINDER_IMG_SIZE)
else:
    print("Synthetic Pathfinder (set PATHFINDER_REAL_DATA_DIR for real data).")
    pf_tr = PathfinderSynthetic(size=2000, img_size=PATHFINDER_IMG_SIZE, seed=1)
    pf_vl = PathfinderSynthetic(size=500,  img_size=PATHFINDER_IMG_SIZE, seed=77)

pf_train_ldr = DataLoader(pf_tr, batch_size=PATHFINDER_BATCH, shuffle=True,  num_workers=0)
pf_val_ldr   = DataLoader(pf_vl, batch_size=PATHFINDER_BATCH, shuffle=False, num_workers=0)

def make_pathfinder_model(f1_type, f2_type):
    return SequenceClassifier(
        vocab_size=2, num_classes=2,
        max_seq_len=PATHFINDER_SEQ_LEN, dim=PATHFINDER_DIM, depth=PATHFINDER_DEPTH,
        attn_factory=lambda d: build_attn_module(d, f1_type, f2_type),
    )

pathfinder_results = run_ablation(
    "Pathfinder", pf_train_ldr, pf_val_ldr, make_pathfinder_model,
    epochs=PATHFINDER_EPOCHS, max_train_batches=PATHFINDER_MAX_TRAIN,
    max_val_batches=PATHFINDER_MAX_VAL, skip_f2=PATHFINDER_SKIP_F2, skip_f1=PATHFINDER_SKIP_F1,
)
display_results("Pathfinder", pathfinder_results, PATHFINDER_SKIP_F2, PATHFINDER_SKIP_F1)

---
## Task 3 — LRA Text (Byte-level IMDb)

Byte-level sentiment classification. Binary. Max seq len: **4096**.

Real data: [LRA GitHub](https://github.com/google-research/long-range-arena) — needs TFDS or LRA preprocessing script.

In [ ]:
class TextSynthetic(Dataset):
    def __init__(self, size=2000, seq_len=1024, seed=0):
        torch.manual_seed(seed)
        self.data   = torch.randint(0, 256, (size, seq_len))
        self.labels = (self.data.float().mean(dim=1) > 128).long()

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], self.labels[i]


def load_text_real(data_dir, split="train", seq_len=4096):
    import csv
    samples = []
    with open(Path(data_dir) / f"imdb_{split}.csv", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ids = list(row["text"].encode("utf-8", errors="replace"))[:seq_len]
            ids += [0] * (seq_len - len(ids))
            samples.append((ids, int(row["label"])))
    class _DS(Dataset):
        def __len__(self): return len(samples)
        def __getitem__(self, i):
            ids, lbl = samples[i]
            return torch.tensor(ids, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)
    return _DS()

print("LRA Text dataset ready.")

In [ ]:
# ── LRA Text config ───────────────────────────────────────────────────────────
TEXT_REAL_DATA_DIR = None   # e.g. "/data/lra/text"
TEXT_SEQ_LEN       = 1024   # use 4096 for full LRA
TEXT_DIM           = 64
TEXT_DEPTH         = 2
TEXT_EPOCHS        = 3
TEXT_BATCH         = 16
TEXT_MAX_TRAIN     = 60
TEXT_MAX_VAL       = 20
TEXT_SKIP_F2       = []
TEXT_SKIP_F1       = []

if TEXT_REAL_DATA_DIR:
    tx_tr = load_text_real(TEXT_REAL_DATA_DIR, "train", TEXT_SEQ_LEN)
    tx_vl = load_text_real(TEXT_REAL_DATA_DIR, "val",   TEXT_SEQ_LEN)
else:
    print("Synthetic Text (set TEXT_REAL_DATA_DIR for real data).")
    tx_tr = TextSynthetic(size=2000, seq_len=TEXT_SEQ_LEN, seed=2)
    tx_vl = TextSynthetic(size=500,  seq_len=TEXT_SEQ_LEN, seed=55)

tx_train_ldr = DataLoader(tx_tr, batch_size=TEXT_BATCH, shuffle=True,  num_workers=0)
tx_val_ldr   = DataLoader(tx_vl, batch_size=TEXT_BATCH, shuffle=False, num_workers=0)

def make_text_model(f1_type, f2_type):
    return SequenceClassifier(
        vocab_size=256, num_classes=2,
        max_seq_len=TEXT_SEQ_LEN, dim=TEXT_DIM, depth=TEXT_DEPTH,
        attn_factory=lambda d: build_attn_module(d, f1_type, f2_type),
    )

text_results = run_ablation(
    "LRA Text", tx_train_ldr, tx_val_ldr, make_text_model,
    epochs=TEXT_EPOCHS, max_train_batches=TEXT_MAX_TRAIN,
    max_val_batches=TEXT_MAX_VAL, skip_f2=TEXT_SKIP_F2, skip_f1=TEXT_SKIP_F1,
)
display_results("LRA Text", text_results, TEXT_SKIP_F2, TEXT_SKIP_F1)

---
## Task 4 — LRA Image (CIFAR-10 as Pixel Sequence)

CIFAR-10 images as sequences of **1024 greyscale pixels**. 10-class classification.

Downloaded automatically via torchvision when `IMAGE_DOWNLOAD=True`.

In [ ]:
class CIFAR10PixelSequence(Dataset):
    """CIFAR-10 as greyscale pixel sequences (1024 tokens, vocab=256)."""
    def __init__(self, root="./data", train=True, download=True):
        from torchvision.datasets import CIFAR10
        import torchvision.transforms as T
        ds = CIFAR10(root=root, train=train, download=download,
                     transform=T.Compose([T.Grayscale(), T.ToTensor()]))
        self.seqs, self.labels = [], []
        for img, lbl in ds:
            self.seqs.append((img.flatten() * 255).long())
            self.labels.append(lbl)

    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        return self.seqs[i], torch.tensor(self.labels[i], dtype=torch.long)


class ImageSynthetic(Dataset):
    def __init__(self, size=2000, seed=0):
        torch.manual_seed(seed)
        self.data   = torch.randint(0, 256, (size, 1024))
        self.labels = torch.randint(0, 10,  (size,))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], self.labels[i]

print("LRA Image dataset ready.")

In [ ]:
# ── LRA Image config ──────────────────────────────────────────────────────────
IMAGE_DOWNLOAD      = True
IMAGE_DATA_ROOT     = "./data"
IMAGE_DIM           = 64
IMAGE_DEPTH         = 2
IMAGE_EPOCHS        = 3
IMAGE_BATCH         = 32
IMAGE_MAX_TRAIN     = 60
IMAGE_MAX_VAL       = 20
IMAGE_SKIP_F2       = []
IMAGE_SKIP_F1       = []

try:
    im_tr = CIFAR10PixelSequence(root=IMAGE_DATA_ROOT, train=True,  download=IMAGE_DOWNLOAD)
    im_vl = CIFAR10PixelSequence(root=IMAGE_DATA_ROOT, train=False, download=IMAGE_DOWNLOAD)
    print(f"CIFAR-10: {len(im_tr)} train / {len(im_vl)} val")
except Exception as e:
    print(f"CIFAR-10 unavailable ({e}), using synthetic data.")
    im_tr = ImageSynthetic(size=2000, seed=3)
    im_vl = ImageSynthetic(size=500,  seed=66)

im_train_ldr = DataLoader(im_tr, batch_size=IMAGE_BATCH, shuffle=True,  num_workers=0)
im_val_ldr   = DataLoader(im_vl, batch_size=IMAGE_BATCH, shuffle=False, num_workers=0)

def make_image_model(f1_type, f2_type):
    return SequenceClassifier(
        vocab_size=256, num_classes=10,
        max_seq_len=1024, dim=IMAGE_DIM, depth=IMAGE_DEPTH,
        attn_factory=lambda d: build_attn_module(d, f1_type, f2_type),
    )

image_results = run_ablation(
    "LRA Image", im_train_ldr, im_val_ldr, make_image_model,
    epochs=IMAGE_EPOCHS, max_train_batches=IMAGE_MAX_TRAIN,
    max_val_batches=IMAGE_MAX_VAL, skip_f2=IMAGE_SKIP_F2, skip_f1=IMAGE_SKIP_F1,
)
display_results("LRA Image", image_results, IMAGE_SKIP_F2, IMAGE_SKIP_F1)

---
## Summary — All Tasks

In [ ]:
all_task_results = {
    "ListOps":    listops_results,
    "Pathfinder": pathfinder_results,
    "LRA Text":   text_results,
    "LRA Image":  image_results,
}

# Print all tables again for easy comparison
for task_name, res in all_task_results.items():
    display_results(task_name, res)

In [ ]:
# ── Best-per-task summary ─────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"{'Task':<15} {'Softmax':>10} {'Best GAM':>10} {'Delta':>8}  Best config")
print(f"{'─'*70}")
for task_name, res in all_task_results.items():
    sm_acc  = res.get(SOFTMAX_KEY, {}).get("val_acc", float("nan"))
    gam     = {k: v["val_acc"] for k, v in res.items() if k != SOFTMAX_KEY}
    if not gam:
        continue
    best_k  = max(gam, key=gam.get)
    best_v  = gam[best_k]
    config  = f"f2={best_k[0]}  f1={best_k[1]}"
    print(f"{task_name:<15} {sm_acc:>10.4f} {best_v:>10.4f} {best_v-sm_acc:>+8.4f}  {config}")